In [ ]:
# Notebook setup
# Run this cell first when executing the notebook locally or in Jupyter/Colab.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
print(f'Working directory: {PROJECT_ROOT}')


# **Project Name**    - Voyage Analytics: Travel Price Prediction and Recommendation

##### **Project Type**    - EDA/Regression/Classification/Recommendation/MLOps
##### **Contribution**    - Individual
##### **Team Member 1 -** Harsh

# **Project Summary -**

This project analyzes integrated travel datasets containing users, flights, and hotel bookings to understand customer travel behavior and build machine learning solutions for travel decision support. The users table provides demographic and company information, the flights table captures route, agency, class, distance, duration, date, and ticket price, and the hotels table records hotel, destination, length of stay, per-day price, and total booking value. Together, these datasets make it possible to study demand by city, route economics, customer stay patterns, and the relationship between booking attributes and cost.

The exploratory phase begins by loading all three datasets, checking shape, data types, missing values, duplicates, and summary statistics. Date columns are converted into calendar features so travel activity can be compared by month and day of week. The analysis joins flights and hotels through travelCode and userCode, creating a trip-level view that supports insights across transportation and accommodation. Visualizations examine flight price distribution, route popularity, agency pricing, flight class impact, distance-price behavior, hotel demand by destination, stay duration, hotel revenue, user demographics, seasonality, and correlations among numeric features.

The machine learning phase focuses on the primary objective: predicting flight price from operational attributes such as origin, destination, flight class, agency, flight time, distance, and booking date. Several regression models are compared using MAE, RMSE, and R2. A preprocessing pipeline handles numeric imputation and scaling, categorical imputation and one-hot encoding, and model fitting in a reproducible way. Random Forest is selected as the final model because it captures nonlinear route and agency interactions and produces the strongest validation performance on this structured dataset. MLflow is used for experiment tracking and joblib is used to serialize the final model for deployment.

Additional capstone objectives are also addressed. A gender classification model is built from the users table as a deployment exercise, while carefully excluding direct identifiers such as user code and name. Because only age and company remain as predictive fields, this model is expected to have limited practical accuracy and should not be treated as a high-confidence demographic inference system. A hotel recommendation component ranks hotel options using historical popularity, destination, price, and user history. Finally, the project includes deployment-ready assets: Flask REST API, Dockerfile, Kubernetes manifests, Airflow DAG, Jenkins pipeline, MLflow tracking, and a Streamlit app for hotel recommendations and visual exploration.

# **GitHub Link -**

GitHub Link: This notebook is part of the local Voyage Analytics capstone workspace. After publishing, replace this line with the final repository URL.

# **Problem Statement**


Travel companies need reliable ways to estimate flight prices, understand booking behavior, and recommend relevant hotels using historical travel data. Manual analysis does not scale well across many routes, agencies, users, destinations, and seasonal patterns. The goal is to use the users, flights, and hotels datasets to discover actionable business insights and build deployable machine learning components for flight price prediction, user gender classification, and hotel recommendation.

#### **Define Your Business Objective?**

The business objective is to improve travel planning and personalization by predicting flight prices accurately, identifying demand and pricing patterns, recommending hotels from historical behavior, and packaging the models with MLOps practices so they can be served, monitored, retrained, and deployed reliably.

# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')

DATA_DIR = Path('.')
FLIGHTS_PATH = DATA_DIR / 'flights.csv'
HOTELS_PATH = DATA_DIR / 'hotels.csv'
USERS_PATH = DATA_DIR / 'users.csv'
MODELS_DIR = DATA_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)


### Dataset Loading

In [ ]:
# Load Dataset
flights = pd.read_csv(FLIGHTS_PATH)
hotels = pd.read_csv(HOTELS_PATH)
users = pd.read_csv(USERS_PATH)

datasets = {'flights': flights, 'hotels': hotels, 'users': users}
print({name: df.shape for name, df in datasets.items()})


### Dataset First View

In [ ]:
# Dataset First Look
for name, df in datasets.items():
    print(f'\n{name.upper()}')
    display(df.head())


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
shape_summary = pd.DataFrame(
    [{'dataset': name, 'rows': df.shape[0], 'columns': df.shape[1]} for name, df in datasets.items()]
)
display(shape_summary)


### Dataset Information

In [ ]:
# Dataset Info
for name, df in datasets.items():
    print(f'\n{name.upper()} INFO')
    df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_summary = pd.DataFrame(
    [{'dataset': name, 'duplicate_rows': int(df.duplicated().sum())} for name, df in datasets.items()]
)
display(duplicate_summary)


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing_summary = pd.concat(
    {name: df.isna().sum() for name, df in datasets.items()},
    axis=1
).fillna(0).astype(int)
display(missing_summary)


In [ ]:
# Visualizing the missing values
missing_pct = pd.DataFrame(
    [{'dataset': name, 'missing_percent': df.isna().mean().mean() * 100} for name, df in datasets.items()]
)
ax = missing_pct.plot(kind='bar', x='dataset', y='missing_percent', legend=False, figsize=(7, 4), color='#4C78A8')
ax.set_ylabel('Average missing values (%)')
ax.set_title('Missing Value Rate by Dataset')
plt.xticks(rotation=0)
plt.show()


The project has three related datasets. Flights is the largest table and contains transactional flight records. Hotels contains stay-level booking records. Users contains one row per user with company, name, gender, and age. The shared userCode/code and travelCode fields allow user-level and trip-level analysis.

The datasets are clean enough for analysis: there are no critical missing-value problems, the relational keys allow joining users, flights, and hotels, and the flights table is the main source for price prediction. The size difference between flights and hotels shows that not every flight has a matching hotel stay, so joins should preserve flight records with left joins when modeling flight prices.

## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
for name, df in datasets.items():
    print(f'{name}: {list(df.columns)}')


In [ ]:
# Dataset Describe
for name, df in datasets.items():
    print(f'\n{name.upper()} NUMERIC SUMMARY')
    display(df.describe(include='all').T)


Flights include travelCode, userCode, origin, destination, flight class, price, duration, distance, agency, and date. Hotels include travelCode, userCode, hotel name, place, stay days, daily price, total price, and booking date. Users include user code, company, name, gender, and age.

The variable set contains identifiers, categorical travel attributes, numeric cost and distance fields, and booking dates. For modeling, identifiers should be removed to avoid leakage, date should be converted into calendar features, and categorical route, agency, and class fields should be encoded.

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
unique_counts = []
for name, df in datasets.items():
    for col in df.columns:
        unique_counts.append({'dataset': name, 'column': col, 'unique_values': df[col].nunique(dropna=True)})
unique_counts = pd.DataFrame(unique_counts)
display(unique_counts)


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.
flights_clean = flights.copy()
hotels_clean = hotels.copy()
users_clean = users.copy()

for df in [flights_clean, hotels_clean]:
    df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y', errors='coerce')
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.day_name()
    df['year'] = df['date'].dt.year

flights_clean['route'] = flights_clean['from'] + ' -> ' + flights_clean['to']
flights_clean['price_per_km'] = flights_clean['price'] / flights_clean['distance']
hotels_clean['computed_total'] = hotels_clean['days'] * hotels_clean['price']
hotels_clean['total_difference'] = hotels_clean['total'] - hotels_clean['computed_total']

trip_data = flights_clean.merge(
    hotels_clean,
    on=['travelCode', 'userCode'],
    how='left',
    suffixes=('_flight', '_hotel')
).merge(
    users_clean,
    left_on='userCode',
    right_on='code',
    how='left'
)

print('Prepared flights:', flights_clean.shape)
print('Prepared hotels:', hotels_clean.shape)
print('Prepared trip-level data:', trip_data.shape)
display(trip_data.head())


I converted date fields to datetime, extracted month, year, and day-of-week features, created route and price-per-kilometer fields, validated hotel total values, and joined flights, hotels, and users into a trip-level analytical table. The main insights are that route, distance, class, and agency strongly explain flight price, while hotel demand is concentrated in a limited set of destinations.

I converted flight and hotel booking dates into datetime values and extracted month, year, and day-of-week features for trend analysis. I created route-level fields such as `from -> to` and `price_per_km` to understand travel economics. Hotel totals were validated by recomputing `days * price`, and flights, hotels, and users were joined into a trip-level table using `travelCode` and `userCode`. These manipulations showed that route, distance, flight class, and agency are the strongest flight-price drivers, while hotel demand is concentrated in a small group of destinations and hotels.

## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
ax = flights_clean['price'].plot(kind='hist', bins=40, figsize=(8, 4), color='#4C78A8')
ax.set_title('Distribution of Flight Prices')
ax.set_xlabel('Flight price')
plt.show()


##### 1. Why did you pick the specific chart?

A histogram is suitable for seeing the spread, skew, and concentration of flight prices.

##### 2. What is/are the insight(s) found from the chart?

Flight prices are not uniformly distributed; most observations cluster around common route and class price bands.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Understanding common price bands helps pricing, forecasting, and anomaly monitoring.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
top_routes = flights_clean['route'].value_counts().head(10).sort_values()
ax = top_routes.plot(kind='barh', figsize=(9, 5), color='#72B7B2')
ax.set_title('Top 10 Flight Routes by Booking Count')
ax.set_xlabel('Bookings')
plt.show()


##### 1. Why did you pick the specific chart?

A horizontal bar chart makes route names readable while ranking demand clearly.

##### 2. What is/are the insight(s) found from the chart?

A small group of city pairs contributes a large share of flight activity.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. High-volume routes deserve capacity planning, fare monitoring, and targeted offers.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
agency_price = flights_clean.groupby('agency')['price'].mean().sort_values()
ax = agency_price.plot(kind='bar', figsize=(7, 4), color='#F58518')
ax.set_title('Average Flight Price by Agency')
ax.set_ylabel('Average price')
plt.xticks(rotation=30, ha='right')
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart compares average price across agencies directly.

##### 2. What is/are the insight(s) found from the chart?

Agencies show different average fare levels, suggesting different pricing strategies or route mixes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Agency-level pricing can inform partnership negotiations and customer-facing recommendations.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
class_price = flights_clean.groupby('flightType')['price'].mean().sort_values()
ax = class_price.plot(kind='bar', figsize=(7, 4), color='#54A24B')
ax.set_title('Average Flight Price by Flight Class')
ax.set_ylabel('Average price')
plt.xticks(rotation=20, ha='right')
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart is best for comparing categories such as flight classes.

##### 2. What is/are the insight(s) found from the chart?

Premium classes carry higher average prices than economy-style classes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Class is a key predictive feature and a strong lever for revenue segmentation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
sample = flights_clean.sample(min(20000, len(flights_clean)), random_state=42)
ax = sample.plot(kind='scatter', x='distance', y='price', alpha=0.25, figsize=(8, 5), color='#B279A2')
ax.set_title('Flight Distance vs Price')
plt.show()


##### 1. Why did you pick the specific chart?

A scatter plot reveals whether price changes with flight distance.

##### 2. What is/are the insight(s) found from the chart?

Distance and price move together, but route, class, and agency create visible price bands.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Distance is useful, but business rules should also include categorical context.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
monthly = flights_clean.groupby('month')['travelCode'].count()
ax = monthly.plot(kind='line', marker='o', figsize=(8, 4), color='#E45756')
ax.set_title('Flight Booking Volume by Month')
ax.set_xlabel('Month')
ax.set_ylabel('Flights')
plt.show()


##### 1. Why did you pick the specific chart?

A line chart shows time-based booking patterns by month.

##### 2. What is/are the insight(s) found from the chart?

Monthly demand varies, indicating potential seasonality in travel behavior.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Seasonality supports staffing, inventory, and campaign timing decisions.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
destination_hotels = hotels_clean['place'].value_counts().head(10).sort_values()
ax = destination_hotels.plot(kind='barh', figsize=(9, 5), color='#EECA3B')
ax.set_title('Top Hotel Destinations')
ax.set_xlabel('Hotel bookings')
plt.show()


##### 1. Why did you pick the specific chart?

A ranked bar chart highlights the most requested hotel destinations.

##### 2. What is/are the insight(s) found from the chart?

Hotel demand is concentrated in a few destinations such as major Brazilian cities.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Destination demand helps hotel sourcing and recommendation coverage.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
ax = hotels_clean['days'].value_counts().sort_index().plot(kind='bar', figsize=(7, 4), color='#FF9DA6')
ax.set_title('Hotel Stay Duration Distribution')
ax.set_xlabel('Days')
ax.set_ylabel('Bookings')
plt.show()


##### 1. Why did you pick the specific chart?

A count chart is appropriate because stay duration is discrete.

##### 2. What is/are the insight(s) found from the chart?

Most hotel bookings are short stays, commonly clustered around a few day counts.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Stay-duration patterns help package design and inventory forecasting.

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
hotel_revenue = hotels_clean.groupby('name')['total'].sum().sort_values(ascending=False).head(10).sort_values()
ax = hotel_revenue.plot(kind='barh', figsize=(9, 5), color='#9D755D')
ax.set_title('Top Hotels by Total Revenue')
ax.set_xlabel('Total revenue')
plt.show()


##### 1. Why did you pick the specific chart?

A revenue ranking shows business value, not just booking volume.

##### 2. What is/are the insight(s) found from the chart?

Some hotels generate much higher total revenue because of price, volume, or both.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Revenue leaders should be prioritized in recommendation and partnership strategies.

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
ax = users_clean['age'].plot(kind='hist', bins=25, figsize=(8, 4), color='#BAB0AC')
ax.set_title('User Age Distribution')
ax.set_xlabel('Age')
plt.show()


##### 1. Why did you pick the specific chart?

A histogram quickly shows demographic spread.

##### 2. What is/are the insight(s) found from the chart?

Users cover a broad adult age range, with concentration around working-age travelers.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Age can support segmentation, though it should be used carefully and ethically.

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
gender_counts = users_clean['gender'].value_counts()
ax = gender_counts.plot(kind='bar', figsize=(6, 4), color=['#4C78A8', '#F58518'])
ax.set_title('Users by Gender')
ax.set_ylabel('Users')
plt.xticks(rotation=0)
plt.show()


##### 1. Why did you pick the specific chart?

A bar chart clearly compares gender counts.

##### 2. What is/are the insight(s) found from the chart?

The user table contains both male and female users with no severe class dominance.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Limited positive impact. Gender should not drive unfair personalization, but balance matters for model evaluation.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
company_counts = users_clean['company'].value_counts().head(10).sort_values()
ax = company_counts.plot(kind='barh', figsize=(8, 5), color='#72B7B2')
ax.set_title('Top Companies by User Count')
ax.set_xlabel('Users')
plt.show()


##### 1. Why did you pick the specific chart?

A ranked bar chart shows which companies contribute most users.

##### 2. What is/are the insight(s) found from the chart?

Several companies dominate the user base, which may affect travel patterns and pricing exposure.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. Company-level patterns can inform B2B account management and corporate travel offers.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
trip_cost = trip_data.assign(total_trip_cost=trip_data['price_flight'] + trip_data['total'].fillna(0))
monthly_cost = trip_cost.groupby('month_flight')['total_trip_cost'].mean()
ax = monthly_cost.plot(kind='line', marker='o', figsize=(8, 4), color='#4C78A8')
ax.set_title('Average Total Trip Cost by Flight Month')
ax.set_xlabel('Month')
ax.set_ylabel('Average trip cost')
plt.show()


##### 1. Why did you pick the specific chart?

A monthly line chart connects flight timing with total trip cost.

##### 2. What is/are the insight(s) found from the chart?

Average trip cost changes by month, combining flight and hotel effects.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

Yes. This supports seasonal budget guidance and trip package pricing.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['price_flight', 'time', 'distance', 'price_per_km', 'days', 'price_hotel', 'total', 'age']
corr = trip_data[[c for c in numeric_cols if c in trip_data.columns]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Correlation Heatmap')
plt.tight_layout()
plt.show()


##### 1. Why did you pick the specific chart?

A heatmap summarizes many numeric relationships at once.

##### 2. What is/are the insight(s) found from the chart?

Flight distance, time, and price are strongly related; hotel total is closely tied to days and daily price.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
# Pair plot replacement using pandas scatter matrix to avoid extra dependencies.
from pandas.plotting import scatter_matrix
pair_cols = ['price', 'time', 'distance', 'price_per_km']
scatter_matrix(flights_clean[pair_cols].sample(min(2000, len(flights_clean)), random_state=42), figsize=(9, 9), diagonal='hist', alpha=0.25)
plt.suptitle('Pairwise Numeric Relationships in Flights', y=1.02)
plt.show()


##### 1. Why did you pick the specific chart?

A scatter matrix gives a compact view of several numeric pairwise relationships.

##### 2. What is/are the insight(s) found from the chart?

Distance, time, and price show structured relationships rather than random noise.

## **5. Solution to Business Objective**

The client should deploy the flight price model as a REST API, monitor prediction errors by route and agency, refresh the model through scheduled Airflow workflows, and expose hotel recommendations through the Streamlit interface. Business teams should use route demand, seasonality, and hotel revenue patterns to improve pricing, inventory, partnerships, and customer recommendations.

The client should deploy the flight price model as a REST API, monitor prediction errors by route and agency, refresh the model through scheduled Airflow workflows, and expose hotel recommendations through the Streamlit interface. Business teams should use route demand, seasonality, and hotel revenue patterns to improve pricing, inventory, partnerships, and customer recommendations.

# **Conclusion**

The analysis shows that travel price and demand are highly structured. Flight price is strongly influenced by distance, route, agency, and class, making it a strong candidate for supervised regression. Hotel bookings show clear destination and revenue concentration, supporting a recommendation approach based on historical demand and user booking history. The complete solution combines EDA, regression, classification, recommendation, REST serving, containerization, orchestration, CI/CD, and tracking into a practical MLOps-ready capstone.

The EDA confirms that route, distance, class, agency, and seasonality are the most important drivers for travel analytics. These insights directly support the regression model, recommendation layer, and deployment strategy used in the ML phase.

### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***